# System B — Glicko-2-PS (beginner version)

**Goal:** Like Elo, but explicitly track uncertainty and allow inactivity to increase uncertainty.

Glicko-2 keeps three numbers per fighter:

- **Rating** $r$: central estimate of skill (like Elo).
- **Rating deviation** $RD$: uncertainty (bigger means “we are less sure”).
- **Volatility** $\sigma$: how quickly the fighter's skill tends to change.

---

## 1) Why Glicko-2 is useful for MMA

MMA careers have:
- long inactivity gaps,
- few fights per year,
- many fighters with only a handful of bouts.

Elo alone doesn't know when a rating is “stale.”
Glicko-2 solves this by increasing $RD$ during inactivity and shrinking updates for very certain fighters.

---

## 2) The math (high level)

Glicko-2 uses an internal scale:

$$
\mu = \frac{r-1500}{173.7178},\qquad
\phi = \frac{RD}{173.7178}
$$

Expected score vs opponent $j$:

$$
g(\phi_j)=\frac{1}{\sqrt{1 + \frac{3\phi_j^2}{\pi^2}}}
$$
$$
E_{ij} = \frac{1}{1 + \exp(-g(\phi_j)(\mu_i-\mu_j))}
$$

---

## 3) Where PS fits (Tier A/B/C)

Glicko-2 needs an observed score $s \in [0,1]$:
- win = 1
- loss = 0
- draw = 0.5 (if you want strict “no delta,” you can skip updates on draws)

With Tier A/B you can use a **slightly shaped** observed score:
- close win ⇒ $s \approx 0.55$
- dominant win ⇒ $s \approx 0.75$
- etc.

But keep shaping conservative.

---

## 4) Inactivity (concept)

Between rating periods, Glicko-2 increases uncertainty:

$$
\phi \leftarrow \sqrt{\phi^2 + \tau^2}
$$

This means:
- after layoffs, ratings become less confident,
- the next fight updates the rating more.


In [ ]:
import sys
from dataclasses import replace
from datetime import date
from importlib import import_module
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

system_b_module = import_module('elo_calculator.application.ranking.system_b_glicko2_ps')
types_module = import_module('elo_calculator.application.ranking.types')
Glicko2PerformanceRatingSystem = system_b_module.Glicko2PerformanceRatingSystem
new_glicko2_state = system_b_module.new_glicko2_state
BoutEvidence = types_module.BoutEvidence
BoutOutcome = types_module.BoutOutcome
EvidenceTier = types_module.EvidenceTier
FinishMethod = types_module.FinishMethod

## 5) Worked toy example (effect of inactivity)

We'll compare two cases:
1) Fighter is active (no long inactivity).
2) Fighter has a layoff ⇒ RD inflates ⇒ next result moves rating more.

We'll keep opponent constant and show the rating change.


In [ ]:
system = Glicko2PerformanceRatingSystem()

opponent_state = replace(new_glicko2_state('opp'), rating=1520.0, rd=90.0, last_mma_date=date(2024, 5, 1))

active_state = replace(new_glicko2_state('fighter_active'), rating=1500.0, rd=80.0, last_mma_date=date(2024, 5, 1))
layoff_state = replace(new_glicko2_state('fighter_layoff'), rating=1500.0, rd=80.0, last_mma_date=date(2022, 5, 1))

evidence_active = BoutEvidence(
    bout_id='glicko_active_case',
    event_date=date(2024, 6, 1),
    sport_key='mma',
    outcome_for_a=BoutOutcome.WIN,
    tier=EvidenceTier.B,
    method=FinishMethod.DEC,
)

evidence_layoff = BoutEvidence(
    bout_id='glicko_layoff_case',
    event_date=date(2024, 6, 1),
    sport_key='mma',
    outcome_for_a=BoutOutcome.WIN,
    tier=EvidenceTier.B,
    method=FinishMethod.DEC,
)

active_pre_rating = active_state.rating
layoff_pre_rating = layoff_state.rating

active_post, _, active_delta, _ = system.update_bout(active_state, opponent_state, evidence_active)
layoff_post, _, layoff_delta, _ = system.update_bout(layoff_state, opponent_state, evidence_layoff)

rows = [
    {
        'case': 'active',
        'pre_rating': round(active_pre_rating, 3),
        'pre_rd': round(active_state.rd, 3),
        'expected_win_prob': round(active_delta.expected_win_prob, 4),
        'delta_rating': round(active_delta.delta_rating, 4),
        'post_rating': round(active_post.rating, 4),
        'post_rd': round(active_post.rd, 4),
        'post_volatility': round(active_post.volatility, 6),
    },
    {
        'case': 'layoff',
        'pre_rating': round(layoff_pre_rating, 3),
        'pre_rd': round(layoff_state.rd, 3),
        'expected_win_prob': round(layoff_delta.expected_win_prob, 4),
        'delta_rating': round(layoff_delta.delta_rating, 4),
        'post_rating': round(layoff_post.rating, 4),
        'post_rd': round(layoff_post.rd, 4),
        'post_volatility': round(layoff_post.volatility, 6),
    },
]

results = pd.DataFrame(rows)
results

## 6) How to use Tier A/B PS shaping in Glicko-2

Instead of using only:
- win = 1, loss = 0

you can use a conservative shaped score:
- close win: 0.55–0.60
- dominant win: 0.70–0.80
- close loss: 0.40–0.45
- dominant loss: 0.20–0.30

This lets the rating respond more to decisive performances **when** you have Tier A/B evidence.

If you want strict “draw = no delta,” you can skip processing draw matches in `matches`.
